In [1]:
import os
import glob
import json

from pathlib import Path
from typing import Generator, Tuple, Optional, List, Tuple, Any

import numpy as np
import cv2
from PIL import Image

import torch
import requests

from PIL import Image, ImageDraw
from transformers import AutoImageProcessor, AutoModelForObjectDetection
import torch
import clip
from boxmot import BotSort, DeepOcSort
from boxmot.utils.ops import letterbox
from ultralytics import YOLO
import torch.nn.functional as F

ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"
WEIGHTS_DIR = ROOT_DIR / "weights"

2025-11-19 23:50:52.730288: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
clip.available_models()

['RN50',
 'RN101',
 'RN50x4',
 'RN50x16',
 'RN50x64',
 'ViT-B/32',
 'ViT-B/16',
 'ViT-L/14',
 'ViT-L/14@336px']

In [2]:
# EMBEDDER_MODEL_FILE = WEIGHTS_DIR / "mobilenet_embedder_v3_small.tflite"
# DETECTOR_MODEL_FILE = WEIGHTS_DIR / "efficientdet_lite2_fl32.tflite"
# TRACKER_MODEL_FILE = WEIGHTS_DIR / "osnet_x0_25_msmt17.pt"
# BaseOptions = mp.tasks.BaseOptions
# ImageEmbedder = mp.tasks.vision.ImageEmbedder
# ImageEmbedderOptions = mp.tasks.vision.ImageEmbedderOptions
# ObjectDetector = mp.tasks.vision.ObjectDetector
# ObjectDetectorOptions = mp.tasks.vision.ObjectDetectorOptions
# VisionRunningMode = mp.tasks.vision.RunningMode

In [ ]:
# # url = "https://images.pexels.com/photos/8413299/pexels-photo-8413299.jpeg?auto=compress&cs=tinysrgb&w=630&h=375&dpr=2"
# # image = Image.open(requests.get(url, stream=True).raw)

# image_path = './bus.jpg'
# img = Image.open(image_path)

# device = "cuda" if torch.cuda.is_available() else "cpu"


# torch.cuda.empty_cache()


# clip_model, preprocess = clip.load("ViT-B/32", device=device)
# yolo_model = YOLO('../weights/yolo11n.pt').to(device)
# tracker = BotSort(
#     reid_weights=Path('../weights/osnet_x0_25_msmt17.pt'),  # Path to ReID model
#     device=0,  # Use CPU for inference
#     half=False
# )

In [3]:
# class ImageEmbedder:
#     def __init__(self, model_path: str = "ViT-B/32",
#     device: str = "cuda"):
#         clip_model, preprocess = clip.load(model_path, device=device)
#         self.clip_model = clip_model
#         self.preprocess = preprocess


#     def embed(self, image: np.ndarray) -> np.ndarray:
#         image = self.preprocess(image).unsqueeze(0).to(self.device)
#         with torch.no_grad():
#             image_features = self.clip_model.encode_image(image)
#         return image_features


# image = preprocess(img).unsqueeze(0).to(device)
# text = clip.tokenize(["a bust", "a human", "a cat"]).to(device)

# with torch.no_grad():
#     image_features = clip_model.encode_image(image)
#     text_features = clip_model.encode_text(text)

#     logits_per_image, logits_per_text = clip_model(image, text)
#     probs = logits_per_image.softmax(dim=-1).cpu().numpy()

# print("Label probs:", probs)  # prints: [[0.9927937  0.00421068 0.00299572]]

In [ ]:
# SAMPLE_PATH = DATA_DIR / "public_test" / "samples" / "BlackBox_0"
# IMAGE_PATH = SAMPLE_PATH / "object_images"

# images = glob.glob(str(IMAGE_PATH / "*.jpg"))
# video_path = SAMPLE_PATH / "drone_video.mp4"

In [11]:
import json
import os


def read_json(filepath):
    """
    Read and parse a JSON file.

    Args:
        filepath (str): Path to the JSON file

    Returns:
        dict/list: Parsed JSON data

    Raises:
        FileNotFoundError: If file doesn't exist
        json.JSONDecodeError: If file contains invalid JSON
    """
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data
    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found")
        raise
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON in '{filepath}': {e}")
        raise
    except Exception as e:
        print(f"Error reading file: {e}")
        raise


def write_json(filepath, data, indent=2, ensure_dir=True):
    """
    Write data to a JSON file.

    Args:
        filepath (str): Path to save the JSON file
        data (dict/list): Data to write
        indent (int): Indentation level for pretty printing (default: 2)
        ensure_dir (bool): Create directory if it doesn't exist (default: True)

    Returns:
        bool: True if successful, False otherwise
    """
    try:
        # Create directory if needed
        if ensure_dir:
            directory = os.path.dirname(filepath)
            if directory and not os.path.exists(directory):
                os.makedirs(directory)

        # Write JSON file
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=indent, ensure_ascii=False)

        return True
    except Exception as e:
        print(f"Error writing to '{filepath}': {e}")
        return False

In [12]:
class ZAICTracker:
    def __init__(
        self,
        yolo_weights,
        clip_model_name="ViT-B/32",
        reid_weights=None,
        image_size: int = 640,
        device="cuda",
    ):
        self.device = device

        print("[INIT] Loading CLIP model...")
        self.clip_model, self.clip_preprocess = clip.load(
            clip_model_name, device=device
        )
        self.clip_model.eval()

        print("[INIT] Loading YOLO model...")
        self.yolo_model = YOLO(yolo_weights).to(device)
        # self.yolo_model = RTDETR("rtdetr-l.pt").to(device)

        print("[INIT] Loading BoT-SORT tracker...")
        self.tracker = BotSort(
            reid_weights=Path(reid_weights) if reid_weights else None,
            device=0 if device == "cuda" else "cpu",
            half=False,
        )
        self.image_size: int = image_size
        self.ref_embs = []

    # ------------------------------------------------------------------
    # Step 1: Reference management
    # ------------------------------------------------------------------
    def add_reference_image(self, image_path):
        """Add one reference image and store its CLIP embedding."""
        image = (
            self.clip_preprocess(
                Image.open(image_path)
                .convert("RGB")
                .resize((self.image_size, self.image_size))
            )
            .unsqueeze(0)
            .to(self.device)
        )
        with torch.no_grad():
            emb = self.clip_model.encode_image(image)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        self.ref_embs.append(emb)
        print(f"[INFO] Added reference image: {image_path}")

    def delete_reference_images(self):
        """Clear all stored reference embeddings."""
        n = len(self.ref_embs)
        self.ref_embs.clear()
        print(f"[🧹] Cleared {n} reference embedding(s).")

    # ------------------------------------------------------------------
    # Step 2: Get crop embedding (RGB only)
    # ------------------------------------------------------------------
    def get_crop_embedding(self, frame_rgb, box):
        x1, y1, x2, y2 = map(int, box)
        crop = frame_rgb[y1:y2, x1:x2, :]  # keep RGB, no channel flip
        image = Image.fromarray(crop)
        tensor = self.clip_preprocess(image).unsqueeze(0).to(self.device)
        with torch.no_grad():
            emb = self.clip_model.encode_image(tensor)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb

    # ------------------------------------------------------------------
    # Step 3: Vectorized similarity
    # ------------------------------------------------------------------
    # def match_object(self, det_emb, threshold=0.3):
    #     """Vectorized cosine-similarity matching with all refs."""
    #     if not self.ref_embs:
    #         return False, 0.0
    #     ref_stack = torch.stack(self.ref_embs, dim=0)  # [N, D]
    #     det_emb = det_emb.unsqueeze(0) if det_emb.ndim == 1 else det_emb  # [1, D]
    #     det_emb = det_emb / det_emb.norm(dim=-1, keepdim=True)
    #     ref_stack = ref_stack / ref_stack.norm(dim=-1, keepdim=True)
    #     sims = F.cosine_similarity(det_emb, ref_stack)
    #     max_sim = sims.max().item()
    #     return max_sim > threshold, max_sim

    def match_object(self, det_emb, threshold=0.3):
        # TODO: use matrix instead of loop
        if not self.ref_embs:
            return False, 0.0
        sims = [
            F.cosine_similarity(det_emb, ref_emb, dim=-1).item()
            for ref_emb in self.ref_embs
        ]
        max_sim = max(sims)
        return max_sim > threshold, max_sim

    # ------------------------------------------------------------------
    # Step 4: Main tracking loop
    # ------------------------------------------------------------------

    def track_video(
        self,
        video_path,
        video_id="unknown",
        threshold=0.5,
        yolo_conf: float = 0.51,
        expand_ratio=1.15,
        output_dir="results",
        debug=True,
    ):
        cap = cv2.VideoCapture(video_path)
        assert cap.isOpened(), f"Cannot open video: {video_path}"

        os.makedirs(output_dir, exist_ok=True)
        frame_idx = 0
        collected_bboxes = []

        while True:
            ret, frame_bgr = cap.read()
            if not ret:
                break
            frame_idx += 1

            h, w = frame_bgr.shape[:2]
            target_height = self.image_size
            scale = target_height / h
            new_w = int(w * scale)
            frame_bgr = cv2.resize(
                frame_bgr, (new_w, target_height), interpolation=cv2.INTER_LINEAR
            )
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            results = self.yolo_model(frame_rgb, verbose=False, imgsz=self.image_size)[
                0
            ]
            detections = results.boxes.xyxy.cpu().numpy()
            confs = results.boxes.conf.cpu().numpy()
            clss = results.boxes.cls.cpu().numpy()

            matched_dets = []

            h, w, _ = frame_rgb.shape

            for box, conf, cls in zip(detections, confs, clss):
                x1, y1, x2, y2 = box

                # Compute center and size
                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2
                bw = (x2 - x1) * expand_ratio
                bh = (y2 - y1) * expand_ratio

                # Recalculate expanded coordinates
                x1e = int(max(0, cx - bw / 2))
                y1e = int(max(0, cy - bh / 2))
                x2e = int(min(w - 1, cx + bw / 2))
                y2e = int(min(h - 1, cy + bh / 2))

                expanded_box = [x1e, y1e, x2e, y2e]

                # Use expanded box for CLIP embedding
                det_emb = self.get_crop_embedding(frame_rgb, expanded_box)
                matched, sim = self.match_object(det_emb, threshold)
                if matched:
                    matched_dets.append(
                        np.concatenate([box, [conf, cls]])
                    )  # keep original box for tracking

            if len(matched_dets) > 0:
                matched_dets = np.array(matched_dets, dtype=np.float32)
                tracks = self.tracker.update(matched_dets, frame_rgb)

                for t in tracks:
                    x1, y1, x2, y2, track_id = map(int, t[:5])
                    collected_bboxes.append(
                        {"frame": frame_idx, "x1": x1, "y1": y1, "x2": x2, "y2": y2}
                    )
                    # Visualization
                    cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(
                        frame_rgb,
                        f"ID {track_id}",
                        (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0, 255, 0),
                        2,
                    )

            # Debug view (ESC to stop)
            if debug:
                cv2.imshow(f"Tracking: {video_id}", frame_rgb)
                if cv2.waitKey(1) & 0xFF == 27:
                    print("[⚠️] Tracking interrupted by user.")
                    break

        cap.release()
        if debug:
            cv2.destroyAllWindows()

        # --- Save results ---
        output_data = {
            "video_id": video_id,
            "detections": [{"bboxes": collected_bboxes}],
        }
        output_path = os.path.join(output_dir, f"{video_id}.json")
        with open(output_path, "w") as f:
            json.dump(output_data, f, indent=2)

        print(f"[✅] Tracking completed. JSON saved to: {output_path}")
        return output_data

In [23]:
import time
import pandas as pd

class ZAICTracker:
    def __init__(
        self,
        yolo_weights,
        image_size: int = 640,
        device="cuda"
    ):
        self.device = device
        self.image_size = image_size

        print("[INIT] Loading YOLO model...")
        self.yolo_model = YOLO(yolo_weights).to(device)

        # Streaming memory
        self.prev_answers = []          # smoothing window
        self.smoothing_window = 5

        # For submission results
        self.predictions = {}           # frame_idx → answer
        self.latency_ms = {}            # frame_idx → infer time
        
    # ------------------------------------------------------------------
    # Pure-YOLO prediction for ONE frame (streaming)
    # ------------------------------------------------------------------
    def predict_streaming(self, frame_rgb_np, frame_idx: int):
        start_t = time.time()

        # YOLO inference
        results = self.yolo_model(
            source=frame_rgb_np,
            imgsz=self.image_size,
            verbose=False
        )[0]

        dets = results.boxes.xyxy.cpu().numpy()
        confs = results.boxes.conf.cpu().numpy()
        clss = results.boxes.cls.cpu().numpy()

        if len(dets) == 0:
            ans = "none"
        else:
            # choose the highest confidence detection
            idx = np.argmax(confs)
            cls_idx = int(clss[idx])
            ans = self.yolo_model.names[cls_idx]

        # ---- smoothing only using past frames ----
        self.prev_answers.append(ans)
        if len(self.prev_answers) > self.smoothing_window:
            self.prev_answers.pop(0)
        
        final_ans = max(set(self.prev_answers), key=self.prev_answers.count)

        # latency (ms)
        elapsed = round((time.time() - start_t) * 1000, 3)

        # store frame result
        self.predictions[frame_idx] = final_ans
        self.latency_ms[frame_idx] = elapsed

        return final_ans, elapsed

    # ------------------------------------------------------------------
    # ORIGINAL track_video updated to use predict_streaming()
    # ------------------------------------------------------------------
    def track_video(
        self,
        video_path,
        video_id="unknown",
        threshold=0.5,        # not used anymore
        yolo_conf: float = 0.51,
        expand_ratio=1.15,    # kept, but no CLIP so this only visual
        output_dir="results",
        debug=True,
    ):
        cap = cv2.VideoCapture(video_path)
        assert cap.isOpened(), f"Cannot open video: {video_path}"

        os.makedirs(output_dir, exist_ok=True)

        frame_idx = 0
        collected_bboxes = []

        while True:
            ret, frame_bgr = cap.read()
            if not ret:
                break

            frame_idx += 1

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            # ---- YOLO streaming prediction ----
            answer, latency = self.predict_streaming(
                frame_rgb, frame_idx
            )

            # ---- get bounding boxes (optional for JSON) ----
            results = self.yolo_model(
                source=frame_rgb,
                imgsz=self.image_size,
                # conf=yolo_conf,
                verbose=False
            )[0]

            dets = results.boxes.xyxy.cpu().numpy()

            for (x1, y1, x2, y2) in dets:
                collected_bboxes.append({
                    "frame": frame_idx,
                    "x1": int(x1),
                    "y1": int(y1),
                    "x2": int(x2),
                    "y2": int(y2)
                })

                # visualization
                if debug:
                    cv2.rectangle(frame_rgb, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)

            if debug:
                cv2.imshow(f"Tracking: {video_id}", frame_rgb)
                if cv2.waitKey(1) & 0xFF == 27:
                    print("[⚠️] Tracking interrupted by user.")
                    break

        cap.release()
        if debug:
            cv2.destroyAllWindows()

        # Save JSON
        output_data = {
            "video_id": video_id,
            "detections": [{"bboxes": collected_bboxes}],
        }
        output_path = os.path.join(output_dir, f"{video_id}.json")
        with open(output_path, "w") as f:
            json.dump(output_data, f, indent=2)

        print(f"[✅] Tracking completed. JSON saved to: {output_path}")
        return output_data

    # ------------------------------------------------------------------
    # Build CSV + JSON submission (like AeroEyes)
    # ------------------------------------------------------------------
    def load_results(
        self,
        csv_path="time_submission.csv",
        json_path="jupyter_submission.json"
    ):
        ids = sorted(self.predictions.keys())

        rows = [
            {"id": fid, "answer": self.predictions[fid], "time": self.latency_ms[fid]}
            for fid in ids
        ]

        pd.DataFrame(rows).to_csv(csv_path, index=False)

        with open(json_path, "w") as f:
            json.dump(rows, f, indent=2)

        print(f"[CSV] Saved to {csv_path}")
        print(f"[JSON] Saved to {json_path}")


In [24]:
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"

tracker = ZAICTracker(
    yolo_weights="../weights/zaic_yolo11n_001.pt",
    # clip_model_name="ViT-L/14@336px",
    # clip_model_name="ViT-B/32",
    # reid_weights="../weights/osnet_x0_25_msmt17.pt",
    device=device,
    image_size=640,
)

[INIT] Loading YOLO model...


In [25]:
os.listdir(str(DATA_DIR / "public_test" / "samples"))

['BlackBox_1',
 'LifeJacket_0',
 'CardboardBox_0',
 'BlackBox_0',
 'LifeJacket_1',
 'CardboardBox_1']

In [22]:
for sample in os.listdir(str(DATA_DIR / "public_test" / "samples")):
    VIDEO_ID = sample
    SAMPLE_PATH = DATA_DIR / "public_test" / "samples" / VIDEO_ID
    IMAGE_PATH = SAMPLE_PATH / "object_images"

    images = glob.glob(str(IMAGE_PATH / "*.jpg"))
    video_path = SAMPLE_PATH / "drone_video.mp4"

    # print(f"Checking {sample} ...")
    # tracker.delete_reference_images()

    # # Step 1: Add reference images
    # for img in images:
    #     tracker.add_reference_image(img)

    # Step 2: Track objects in video
    tracker.track_video(
        video_path,
        threshold=0.45,
        # yolo_conf=0.35,
        video_id=VIDEO_ID,
        output_dir="../temp/results/19112025/001",
        debug=False,
    )


TypeError: ZAICTracker.predict_streaming() got an unexpected keyword argument 'yolo_conf'

In [10]:
result_files = glob.glob("../temp/results/19112025/001/*.json")
results = []
for file in result_files:
    print(file)
    result = read_json(file)
    results.append(result)


len(results)

../temp/results/19112025/001/CardboardBox_1.json


NameError: name 'read_json' is not defined

In [ ]:
write_json(filepath="../temp/results/19112026_final_results_001.json", data=results)

True

In [ ]:
VIDEO_ID = "CardboardBox_0"
SAMPLE_PATH = DATA_DIR / "public_test" / "samples" / VIDEO_ID
IMAGE_PATH = SAMPLE_PATH / "object_images"

images = glob.glob(str(IMAGE_PATH / "*.jpg"))
video_path = SAMPLE_PATH / "drone_video.mp4"
tracker.delete_reference_images()

# Step 1: Add reference images
for img in images:
    tracker.add_reference_image(img)


# Step 2: Start video tracking
tracker.track_video(
    video_path, threshold=0.65, video_id=VIDEO_ID, output_dir="../temp/results"
)

[🧹] Cleared 3 reference embedding(s).
[INFO] Added reference image: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/public_test/samples/CardboardBox_0/object_images/img_2.jpg
[INFO] Added reference image: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/public_test/samples/CardboardBox_0/object_images/img_3.jpg
[INFO] Added reference image: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/public_test/samples/CardboardBox_0/object_images/img_1.jpg
